# Rename Legacy BigQuery Tables

This notebook renames legacy tables to `{name}_legacy` using a safe BigQuery pattern:
1. `CREATE TABLE ... AS SELECT * ...` (copy)
2. Optional row-count check
3. `DROP TABLE` original

Target mappings:
- `tree` -> `tree_legacy`
- `quarter_section` -> `quarter_section_legacy`
- `service_request` -> `service_request_legacy`
- `address` -> `address_legacy` (if present)

In [4]:
from google.cloud import bigquery
import os
from pathlib import Path

# Load .env files if python-dotenv is installed.
try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

if load_dotenv is not None:
    load_dotenv(Path.cwd() / ".env", override=False)
    load_dotenv(Path.cwd() / "database" / ".env", override=False)
    load_dotenv(Path.cwd() / "database" / "cloud_functions" / ".env", override=False)

PROJECT_ID = os.getenv("BQ_PROJECT_ID", "mke-trees")
DATASET = os.getenv("BQ_DATASET", "mke_tree_dataset")
LOCATION = os.getenv("BQ_LOCATION", "us-central1")

# Credential lookup order:
# 1) GOOGLE_APPLICATION_CREDENTIALS from env
# 2) serviceAccountKey.json in common local paths
candidate_paths = []
if os.getenv("GOOGLE_APPLICATION_CREDENTIALS"):
    candidate_paths.append(Path(os.getenv("GOOGLE_APPLICATION_CREDENTIALS")))

candidate_paths.extend(
    [
        Path.cwd() / "serviceAccountKey.json",
        Path.cwd() / "database" / "serviceAccountKey.json",
        Path.cwd() / "database" / "cloud_functions" / "serviceAccountKey.json",
    ]
)

cred_path = None
for p in candidate_paths:
    if p and p.is_file():
        cred_path = p.resolve()
        break

if cred_path is not None:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(cred_path)
else:
    raise FileNotFoundError(
        "No service account key found. Put serviceAccountKey.json in database/ or "
        "database/cloud_functions/, or set GOOGLE_APPLICATION_CREDENTIALS to its full path."
    )

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

print(f"Using project={PROJECT_ID}, dataset={DATASET}, location={LOCATION}")
print(f"Credentials: {os.environ['GOOGLE_APPLICATION_CREDENTIALS']}")

Using project=mke-trees, dataset=mke_tree_dataset, location=us-central1
Credentials: C:\Users\edex\OneDrive - Milwaukee School of Engineering\Desktop\AI Club\tree-frontend\trees-frontend\database\serviceAccountKey.json


In [10]:
TABLE_RENAMES = [
    ("tree", "tree_legacy"),
    ("quarter_section", "quarter_section_legacy"),
    ("service_request", "service_request_legacy"),
    ("address", "address_legacy"),
]

# Safety switches
DROP_SOURCE_TABLES = True  # Set True to copy + drop (rename behavior)
FAIL_IF_TARGET_EXISTS = False  # Set True if you want strict behavior

In [11]:
def fqn(table_name: str) -> str:
    return f"{PROJECT_ID}.{DATASET}.{table_name}"


def table_exists(table_name: str) -> bool:
    try:
        client.get_table(fqn(table_name))
        return True
    except Exception:
        return False


def row_count(table_name: str) -> int:
    sql = f"SELECT COUNT(*) AS c FROM `{fqn(table_name)}`"
    return int(list(client.query(sql).result())[0]["c"])


def run_sql(sql: str):
    return client.query(sql).result()

In [12]:
results = []

for source, target in TABLE_RENAMES:
    source_exists = table_exists(source)
    target_exists = table_exists(target)

    if not source_exists:
        results.append({
            "source": source,
            "target": target,
            "status": "skipped_source_missing",
        })
        continue

    if target_exists and FAIL_IF_TARGET_EXISTS:
        raise RuntimeError(f"Target already exists: {fqn(target)}")

    if not target_exists:
        copy_sql = f"""
        CREATE TABLE `{fqn(target)}` AS
        SELECT * FROM `{fqn(source)}`
        """
        run_sql(copy_sql)

    src_count = row_count(source)
    tgt_count = row_count(target)
    if src_count != tgt_count:
        raise RuntimeError(
            f"Row-count mismatch for {source} -> {target}: {src_count} vs {tgt_count}"
        )

    dropped = False
    if DROP_SOURCE_TABLES:
        drop_sql = f"DROP TABLE `{fqn(source)}`"
        run_sql(drop_sql)
        dropped = True

    results.append({
        "source": source,
        "target": target,
        "status": "renamed" if dropped else "copied_only",
        "source_count": src_count,
        "target_count": tgt_count,
        "source_dropped": dropped,
    })

results

[{'source': 'tree',
  'target': 'tree_legacy',
  'status': 'renamed',
  'source_count': 191074,
  'target_count': 191074,
  'source_dropped': True},
 {'source': 'quarter_section',
  'target': 'quarter_section_legacy',
  'status': 'renamed',
  'source_count': 696,
  'target_count': 696,
  'source_dropped': True},
 {'source': 'service_request',
  'target': 'service_request_legacy',
  'status': 'renamed',
  'source_count': 0,
  'target_count': 0,
  'source_dropped': True},
 {'source': 'address',
  'target': 'address_legacy',
  'status': 'renamed',
  'source_count': 0,
  'target_count': 0,
  'source_dropped': True}]

In [14]:
verify_sql = f"""
SELECT table_name
FROM `{PROJECT_ID}.{DATASET}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

rows = list(client.query(verify_sql).result())
print(f"Found {len(rows)} matching tables:")
for r in rows:
    print(f"- {r['table_name']}")

Found 15 matching tables:
- address_legacy
- qs_priority
- quarter_section_legacy
- quarter_sections
- service_request_assignees
- service_request_legacy
- service_requests
- shap
- species
- tree_inspections
- tree_legacy
- trees_core
- trees_features
- users
- vw_qs_analytics_new
